<a href="https://colab.research.google.com/github/oxanaadry-svg/python_sis_26_2/blob/oxana_silva/aula_11/oxana_silva_aula11_exercicio1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

# Instala a biblioteca utilizada para criar planilhas Excel
%pip install -q openpyxl

# Importa as bibliotecas utilizadas no programa
from pathlib import Path
import xml.etree.ElementTree as et
from openpyxl import Workbook


# Cria a pasta onde serao armazenadas as notas fiscais
pasta = Path("notas_xml")
pasta.mkdir(exist_ok=True)


# Funcao responsavel por criar os arquivos XML
def criar_notas():

    # Lista com os dados das notas fiscais
    notas = [
        ("1001", "2026-10-10", "Tech Solutions LTDA",
         "12.345.678/0001-90", "4500.00"),

        ("1002", "2026-10-12", "Cloud Cloud Servicos",
         "98.765.432/0001-11", "1280.50"),

        ("1003", "2026-10-15", "Hardware & Redes SA",
         "45.123.890/0001-55", "8900.00"),
    ]

    # Percorre os dados de cada nota fiscal
    for numero, data, fornecedor, cnpj, valor in notas:

        # Cria o elemento principal do XML
        raiz = et.Element("nota")

        # Cria as tags com os dados da nota
        et.SubElement(raiz, "numero").text = numero
        et.SubElement(raiz, "data").text = data
        et.SubElement(raiz, "fornecedor").text = fornecedor
        et.SubElement(raiz, "cnpj").text = cnpj
        et.SubElement(raiz, "valor").text = valor

        # Salva o arquivo XML na pasta
        arquivo = pasta / f"nota_{numero}.xml"

        et.ElementTree(raiz).write(
            arquivo,
            encoding="utf-8",
            xml_declaration=True
        )

        print(f"Nota {numero} criada com sucesso!")


# Funcao responsavel por ler os arquivos XML
def ler_notas():

    # Cria uma lista para guardar os dados das notas
    dados_notas = []

    # Procura todos os arquivos XML da pasta
    for arquivo in sorted(pasta.glob("*.xml")):

        try:

            # Faz a leitura do arquivo XML
            raiz = et.parse(arquivo).getroot()

            # Guarda os dados encontrados nas tags
            dados = []

            campos = ["numero", "data", "fornecedor", "cnpj", "valor"]

            for campo in campos:

                # Verifica se a informacao existe
                texto = raiz.findtext(campo)

                if texto is None or not texto.strip():
                    raise ValueError(f"Campo ausente: {campo}")

                dados.append(texto.strip())

            # Converte o valor para numero decimal
            valor = float(dados[4])

            # Adiciona os dados na lista
            dados_notas.append([
                dados[0],
                dados[1],
                dados[2],
                dados[3],
                valor
            ])

            print(f"Nota {dados[0]} lida com sucesso!")

        except (et.ParseError, OSError, ValueError) as erro:

            # Mostra o erro caso o arquivo esteja incorreto
            print(f"Erro no arquivo {arquivo.name}: {erro}")

    return dados_notas


# Funcao responsavel por criar a planilha Excel
def criar_planilha(dados_notas):

    # Cria uma nova planilha
    planilha = Workbook()

    # Seleciona a primeira aba
    aba = planilha.active

    # Define o nome da aba
    aba.title = "Notas Fiscais"

    # Cria o cabecalho da planilha
    aba.append([
        "Numero",
        "Data",
        "Fornecedor",
        "CNPJ",
        "Valor Total"
    ])

    # Variavel para somar os valores das notas
    total = 0

    # Percorre os dados das notas fiscais
    for dados in dados_notas:

        # Adiciona os dados na planilha
        aba.append(dados)

        # Formata a coluna de valores
        aba.cell(
            row=aba.max_row,
            column=5
        ).number_format = '"R$" #,##0.00'

        # Soma o valor da nota
        total += dados[4]

    # Adiciona o total geral na planilha
    aba.append(["", "", "", "TOTAL GERAL", total])

    # Formata o valor total
    aba.cell(
        row=aba.max_row,
        column=5
    ).number_format = '"R$" #,##0.00'

    # Ajusta a largura das colunas
    aba.column_dimensions["A"].width = 15
    aba.column_dimensions["B"].width = 18
    aba.column_dimensions["C"].width = 30
    aba.column_dimensions["D"].width = 23
    aba.column_dimensions["E"].width = 18

    # Salva a planilha Excel
    planilha.save("notas_fiscais.xlsx")

    print("\nPlanilha criada com sucesso!")
    print(f"Quantidade de notas: {len(dados_notas)}")
    print(f"Valor total das notas: R$ {total:.2f}")


# Funcao principal do programa
def principal():

    print("===================================")
    print("     SISTEMA DE NOTAS FISCAIS")
    print("===================================")

    # Etapa 1 - Criacao dos arquivos XML
    print("\nEtapa 1 - Criando arquivos XML...")
    criar_notas()

    # Etapa 2 - Leitura dos arquivos XML
    print("\nEtapa 2 - Lendo arquivos XML...")
    dados = ler_notas()

    # Etapa 3 - Criacao da planilha Excel
    print("\nEtapa 3 - Criando planilha Excel...")
    criar_planilha(dados)

    print("\n===================================")
    print("       PROCESSO FINALIZADO!")
    print("===================================")


# Executa o programa
principal()


     SISTEMA DE NOTAS FISCAIS

Etapa 1 - Criando arquivos XML...
Nota 1001 criada com sucesso!
Nota 1002 criada com sucesso!
Nota 1003 criada com sucesso!

Etapa 2 - Lendo arquivos XML...
Nota 1001 lida com sucesso!
Nota 1002 lida com sucesso!
Nota 1003 lida com sucesso!

Etapa 3 - Criando planilha Excel...

Planilha criada com sucesso!
Quantidade de notas: 3
Valor total das notas: R$ 14680.50

       PROCESSO FINALIZADO!
